# Efficienza: il modello che si addestra non è quello che si usa

Il codice del capitolo [«Efficienza: il modello che si addestra non è quello che si usa»](https://book.paithon.it/main/Efficienza/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Efficienza: il modello che si addestra non è quello che si usa

[Leggi la pagina](https://book.paithon.it/main/Efficienza/overview.html)


### Il problema, in due numeri


In [ ]:
PARAMETRI = 7_000_000_000     # sette miliardi di parametri
SCHEDA_GB = 16                # la scheda grafica ha sedici gigabyte di memoria

print(f"{'formato':<9} {'bit':>4} {'peso':>10}   ci sta nella scheda?")
for nome, bit in (("float32", 32), ("float16", 16), ("int8", 8), ("int4", 4)):
    gb = PARAMETRI * bit / 8 / 1e9
    print(f"{nome:<9} {bit:>4} {gb:>7.1f} GB   {'sì' if gb < SCHEDA_GB else 'no'}")

## Meno bit: che cosa si perde arrotondando

[Leggi la pagina](https://book.paithon.it/main/Efficienza/meno-bit.html)


### Quanti bit servono davvero


In [ ]:
import torch

torch.manual_seed(0)
# un thread solo: due esecuzioni di fila danno lo stesso numero. Su un'altra
# macchina le ultime cifre ballano, perche' cambia l'ordine delle somme
torch.set_num_threads(1)


def quantizza(t, bit, gruppo=None):
    """Porta i numeri su 2**bit livelli interi e li riporta indietro.

    Con `gruppo` la scala non e' una per tutto il tensore, ma una ogni
    `gruppo` numeri consecutivi lungo l'ultima dimensione."""
    q = 2 ** (bit - 1) - 1
    if gruppo is None:
        s = t.abs().max()
        return torch.round(t / s * q).clamp(-q - 1, q) * s / q
    f = t.reshape(*t.shape[:-1], -1, gruppo)
    # il `clamp` sul denominatore non e' pedanteria: se un gruppo e' tutto di
    # zeri la scala vale zero e la divisione restituisce `nan` senza avvisare.
    # Non succede sui pesi di una rete addestrata; succede eccome su una rete
    # potata, che e' esattamente la cosa che il capitolo invita a comporre
    s = f.abs().amax(-1, keepdim=True).clamp(min=1e-12)
    return (torch.round(f / s * q).clamp(-q - 1, q) * s / q).reshape(t.shape)


W = torch.randn(256, 512)
x = torch.randn(512, 64)
vero = W @ x


def errore(Wq):
    """Di quanto cambia il risultato, in percentuale."""
    return ((Wq @ x - vero).norm() / vero.norm() * 100).item()


print(f"{'bit':>4} {'una scala per tutto':>21} {'una scala ogni 64 pesi':>24}")
for bit in (8, 6, 4, 3):
    print(f"{bit:>4} {errore(quantizza(W, bit)):>20.2f}% "
          f"{errore(quantizza(W, bit, 64)):>23.2f}%")

# i due massimi da cui passa il conto del modello uniforme
sigma = W.std()
gruppi = W.reshape(256, -1, 64).abs().amax(-1)
print(f"massimo di tutta la matrice: {W.abs().max() / sigma:.2f} sigma")
print(f"massimo di un gruppo da 64, in media quadratica: "
      f"{gruppi.pow(2).mean().sqrt() / sigma:.2f} sigma")

### Le poche componenti enormi


In [ ]:
X = torch.randn(64, 512)
enormi = [7, 133, 401]            # tre componenti su 512, lo 0,6 per cento
X[:, enormi] *= 60
atteso = X @ W.T


def errore_x(Xq):
    return ((Xq @ W.T - atteso).norm() / atteso.norm() * 100).item()


resto = [i for i in range(512) if i not in enormi]
misto = X.clone()
misto[:, resto] = quantizza(X[:, resto], 8)   # la scala si calcola senza le enormi

print(f"la componente normale piu' grande vale {X[:, resto].abs().max():.1f}")
print(f"la componente enorme piu' grande vale  {X[:, enormi].abs().max():.1f}")
print()
print(f"8 bit, una scala per tutto:           {errore_x(quantizza(X, 8)):6.2f}%")
print(f"8 bit, ma le tre enormi tenute intere: {errore_x(misto):6.2f}%")

## Meno pesi: la promessa che si riscuote male

[Leggi la pagina](https://book.paithon.it/main/Efficienza/meno-pesi.html)


### Quali pesi si tolgono


In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
# un thread solo: due esecuzioni di fila danno lo stesso numero. Su un'altra
# macchina le ultime cifre ballano, perche' cambia l'ordine delle somme
torch.set_num_threads(1)

dati = load_digits()
Xtr, Xte, ytr, yte = train_test_split(dati.data / 16.0, dati.target,
                                      test_size=0.3, random_state=0)
Xtr = torch.tensor(Xtr, dtype=torch.float32)
Xte = torch.tensor(Xte, dtype=torch.float32)
ytr, yte = torch.tensor(ytr), torch.tensor(yte)


def costruisci():
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(64, 256), nn.ReLU(),
                         nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))


def addestra(rete, passi, maschere=None):
    """Se ci sono le maschere, i pesi tagliati vengono rimessi a zero
    dopo ogni passo: l'ottimizzatore non puo' farli risorgere."""
    opt = torch.optim.Adam(rete.parameters(), lr=1e-3)
    matrici = [p for p in rete.parameters() if p.dim() == 2]
    for _ in range(passi):
        nn.functional.cross_entropy(rete(Xtr), ytr).backward()
        opt.step()
        opt.zero_grad()
        if maschere:
            with torch.no_grad():
                for p, m in zip(matrici, maschere):
                    p *= m
    return rete


def accuratezza(rete):
    with torch.no_grad():
        return (rete(Xte).argmax(1) == yte).float().mean().item() * 100


rete = addestra(costruisci(), 600)
pieni = [p.detach().clone() for p in rete.parameters()]
print(f"rete intera: {accuratezza(rete):.1f}%")
print()
print(f"{'tolti':>7} {'subito dopo':>13} {'dopo il riaddestramento':>25}")
for frazione in (.5, .8, .9, .95):
    with torch.no_grad():
        for p, originale in zip(rete.parameters(), pieni):
            p.copy_(originale)                 # si riparte sempre dalla rete intera
        maschere = []
        for p in [q for q in rete.parameters() if q.dim() == 2]:
            soglia = p.abs().flatten().kthvalue(int(frazione * p.numel())).values
            m = (p.abs() >= soglia).float()
            p *= m
            maschere.append(m)
    subito = accuratezza(rete)
    dopo = accuratezza(addestra(rete, 300, maschere))
    print(f"{frazione*100:>6.0f}% {subito:>12.1f}% {dopo:>24.1f}%")

### Perché il conto non si accorge degli zeri


In [ ]:
torch.manual_seed(0)
# nomi tutti nuovi: nel notebook compagno le pagine si susseguono nello stesso
# spazio dei nomi, e riusare `W` costringerebbe a rieseguirle sempre in ordine
piena = torch.randn(1024, 1024)
soglia = piena.abs().flatten().kthvalue(int(0.95 * piena.numel())).values
rada = piena * (piena.abs() >= soglia)
ingressi = torch.randn(1024, 256)

zeri = (rada == 0).float().mean().item()
print(f"zeri nella matrice rada: {zeri * 100:.0f}%")
print(f"moltiplicazioni che servirebbero: una su {1 / (1 - zeri):.0f}")
print(f"moltiplicazioni che il calcolatore fa, in tutti e due i casi: "
      f"{piena.numel() * ingressi.shape[1] / 1e9:.2f} miliardi")

### Un chip che guarda i numeri


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 512
W = rng.normal(size=(n, n)) * (rng.random((n, n)) < 0.1)   # potata: nove pesi su dieci a zero
a = np.maximum(rng.normal(size=n), 0)                      # dopo una ReLU: circa metà a zero

# per ogni colonna i soli pesi non nulli, ciascuno con quante righe vuote lo
# precedono scritto in 4 bit: più di 15 righe vuote di fila vogliono uno zero di riempimento
colonne = []
for j in range(n):
    voci, ultima = [], -1
    for i in np.flatnonzero(W[:, j]):
        while i - ultima > 16:
            ultima += 16
            voci.append((ultima, 0.0))
        voci.append((i, W[i, j]))
        ultima = i
    colonne.append(voci)

o, moltiplicazioni, a_vuoto = np.zeros(n), 0, 0
for j in np.flatnonzero(a):                  # gli zeri dell'ingresso non partono nemmeno
    for i, w in colonne[j]:
        o[i] += w * a[j]
        moltiplicazioni += 1
        a_vuoto += w == 0                    # un riempimento si moltiplica come gli altri
print("uguale al prodotto denso:", np.allclose(o, W @ a))
print(f"pesi nulli: {np.mean(W == 0):.0%}, ingressi nulli: {np.mean(a == 0):.0%}")
print(f"moltiplicazioni: {n * n} nel prodotto denso, {moltiplicazioni} saltando gli zeri"
      f" ({moltiplicazioni / (n * n):.1%}), {a_vuoto} delle quali sugli zeri di riempimento")
voci = sum(len(c) for c in colonne)
riempimento = sum(1 for c in colonne for _, w in c if w == 0)
print(f"voci del formato compresso: {voci}, di cui {riempimento} zeri di riempimento")

### Spegnere invece di saltare


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 100_000
# le pause di un'unità di calcolo, in cicli: nove su dieci brevi, una lunga
lunga = rng.random(n) < 0.1
due_specie = np.where(lunga, rng.geometric(1 / 400, n), rng.geometric(1 / 4, n))
# stessa media, ma ogni ciclo fermo ha la stessa probabilità di essere l'ultimo
senza_memoria = rng.geometric(1 / due_specie.mean(), n)

fuga = 1.0       # dispersione in un ciclo passato acceso e fermo
costo = 30.0     # energia per staccare l'unità e riattaccarla
print(f"pausa media {due_specie.mean():.2f} cicli,"
      f" pareggio {costo / fuga:.0f} cicli")

def risparmio(pause, attesa):
    """Stacca dopo `attesa` cicli fermi: dispersione risparmiata, al netto."""
    staccate = pause > attesa
    netto = fuga * (pause[staccate] - attesa).sum() - costo * staccate.sum()
    return netto / (fuga * pause.sum()), staccate.mean()

for nome, pause in [("due specie", due_specie),
                    ("senza memoria", senza_memoria)]:
    print(nome)
    for attesa in [0, 10, 30, 100]:
        r, quota = risparmio(pause, attesa)
        print(f"  attesa {attesa:3d}: risparmiato {r:6.1%} della dispersione,"
              f" staccate {quota:6.1%} delle pause")
    migliore = max(range(301), key=lambda a: risparmio(pause, a)[0])
    print(f"  attesa migliore fra 0 e 300 cicli: {migliore}")

### Le due leve insieme


In [ ]:
def stato_pieno():
    """Rimette la rete com'era dopo il primo addestramento."""
    with torch.no_grad():
        for p, originale in zip(rete.parameters(), pieni):
            p.copy_(originale)


def arrotonda(bit=4, gruppo=64):
    with torch.no_grad():
        for p in rete.parameters():
            if p.dim() == 2:
                p.copy_(quantizza(p, bit, gruppo))     # dalla sezione di prima


stato_pieno()
print(f"rete intera:                {accuratezza(rete):.1f}%")
stato_pieno()
arrotonda()
print(f"solo quattro bit:           {accuratezza(rete):.1f}%")
stato_pieno()
maschere = []
for p in [q for q in rete.parameters() if q.dim() == 2]:
    soglia = p.abs().flatten().kthvalue(int(0.9 * p.numel())).values
    m = (p.abs() >= soglia).float()
    p.data *= m
    maschere.append(m)
addestra(rete, 300, maschere)
print(f"solo potata al 90%:         {accuratezza(rete):.1f}%")
arrotonda()
print(f"potata e poi a quattro bit: {accuratezza(rete):.1f}%")

## Un modello piccolo che imita: la distillazione

[Leggi la pagina](https://book.paithon.it/main/Efficienza/un-modello-piccolo-che-imita.html)


### L’esperimento


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# un thread solo: due esecuzioni di fila danno lo stesso numero. Su un'altra
# macchina le ultime cifre ballano, perche' cambia l'ordine delle somme
torch.set_num_threads(1)

dati = load_digits()
X, Xte, y, yte = train_test_split(dati.data / 16.0, dati.target,
                                  test_size=0.5, random_state=0)
X = torch.tensor(X, dtype=torch.float32)
Xte = torch.tensor(Xte, dtype=torch.float32)
y, yte = torch.tensor(y), torch.tensor(yte)

POCHI = 120                      # le sole etichette che lo studente puo' vedere
Xpoche, ypoche = X[:POCHI], y[:POCHI]
TEMPERATURA = 4.0


def crea(taglie, seme):
    torch.manual_seed(seme)
    strati = []
    for dentro, fuori in zip(taglie, taglie[1:]):
        strati += [nn.Linear(dentro, fuori), nn.ReLU()]
    return nn.Sequential(*strati[:-1])       # l'ultima ReLU non serve


def accuratezza(modello):
    with torch.no_grad():
        return (modello(Xte).argmax(1) == yte).float().mean().item() * 100


def parametri(modello):
    return sum(p.numel() for p in modello.parameters())


maestro = crea([64, 512, 512, 10], seme=0)
opt = torch.optim.Adam(maestro.parameters(), lr=1e-3)
for _ in range(900):
    F.cross_entropy(maestro(X), y).backward()
    opt.step()
    opt.zero_grad()
print(f"maestro, con tutte le {len(X)} etichette: {accuratezza(maestro):.1f}%")
with torch.no_grad():
    logit_maestro = maestro(X)

with torch.no_grad():
    logit_su_pochi = maestro(Xpoche)


def morbida(uscita, bersaglio):
    """Quanto lo studente si discosta dai dubbi del maestro. Il fattore T*T
    rimette il termine morbido sulla scala di quello duro."""
    T = TEMPERATURA
    return F.kl_div(F.log_softmax(uscita / T, dim=1),
                    F.softmax(bersaglio / T, dim=1),
                    reduction="batchmean") * T * T


# tre condizioni, che servono a separare due cose che di solito si confondono:
# i dubbi del maestro, e il fatto che il maestro possa commentare esempi di cui
# lo studente non ha l'etichetta
for etichetta, maestro_su in (("niente maestro", None),
                              ("maestro sui soli 120", "pochi"),
                              ("maestro su tutti gli 898", "tutti")):
    prove = []
    for seme in (1, 2, 3):
        studente = crea([64, 16, 10], seme)
        opt = torch.optim.Adam(studente.parameters(), lr=3e-3)
        for _ in range(900):
            perdita = F.cross_entropy(studente(Xpoche), ypoche)
            if maestro_su == "pochi":
                perdita = 0.3 * perdita + 0.7 * morbida(studente(Xpoche),
                                                        logit_su_pochi)
            elif maestro_su == "tutti":
                perdita = 0.3 * perdita + 0.7 * morbida(studente(X),
                                                        logit_maestro)
            perdita.backward()
            opt.step()
            opt.zero_grad()
        prove.append(accuratezza(studente))
    media = sum(prove) / len(prove)
    print(f"studente, {etichetta:<22} {media:.1f}%   "
          f"(tre semi: {', '.join(f'{p:.1f}' for p in prove)})")
print(f"lo studente ha {parametri(maestro) / parametri(studente):.0f} volte "
      f"meno parametri del maestro")

## Starci non è rispondere: la mappa dell’altra metà

[Leggi la pagina](https://book.paithon.it/main/Efficienza/far-rispondere-in-fretta.html)


### Perché rispondere è un problema di traffico


In [ ]:
N = 4096                       # la larghezza di uno strato
BIT = 16                       # ogni peso in sedici bit

print(f"{'cose insieme':>13} {'conti':>16} {'byte letti':>12} {'conti per byte':>16}")
for k in (1, 4, 16, 64, 256):
    conti = 2 * N * N * k      # per ogni casella, n moltiplicazioni e n somme
    byte = N * N * BIT / 8     # i pesi si leggono una volta sola
    print(f"{k:>13} {conti/1e6:>11.0f} milioni {byte/1e6:>9.0f} MB "
          f"{conti/byte:>16.1f}")